# Pipelines de Pré-processamento - Parte 1
## Fundamentos e Conceitos Básicos

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 10 - Parte 1 de 2
---



## 1. Introdução: Por que Pipelines são Importantes?

Até agora em nosso curso, aprendemos diversas técnicas de pré-processamento:
- Análise estatística e exploratória (Aula 4)
- Tratamento de valores ausentes (Aula 5)
- Detecção e tratamento de outliers (Aula 6)
- Normalização e padronização de dados (Aula 7)
- Codificação de variáveis categóricas (Aula 8)
- Engenharia de features (Aula 9)

No entanto, aplicar essas técnicas manualmente a cada execução de código apresenta alguns desafios:

1. **Reprodutibilidade**: É difícil garantir que o mesmo processo seja aplicado exatamente da mesma forma cada vez
2. **Organização**: O código tende a ficar disperso e desorganizado
3. **Vazamento de dados**: Risco de aplicar transformações incorretamente, levando a resultados tendenciosos
4. **Manutenção**: Atualizar o processo exige alterações em vários locais do código
5. **Produção**: Dificulta a implementação em ambientes produtivos

Os pipelines de dados resolvem estes problemas ao organizar o fluxo de pré-processamento em uma sequência bem definida de operações.

## 2. O que é um Pipeline de Dados?

Um pipeline de dados é uma sequência organizada de etapas de processamento de dados que transforma dados brutos em um formato adequado para análise ou modelagem. Cada etapa do pipeline recebe a saída da etapa anterior como entrada.

pipeline-svg__.svg

Benefícios dos pipelines:

- **Automação**: Execução automática de todas as etapas de processamento
- **Consistência**: Garantia que os mesmos passos sejam aplicados a novos dados
- **Modularidade**: Facilidade para adicionar, remover ou substituir etapas
- **Encapsulamento**: Organização de lógica complexa em componentes gerenciáveis
- **Validação**: Facilidade para validar o processo como um todo

## 3. Componentes de um Pipeline

Um pipeline típico de ciência de dados inclui vários componentes:

1. **Aquisição de dados**: Obtenção dos dados de diferentes fontes
2. **Limpeza de dados**: Tratamento de valores ausentes, outliers e erros
3. **Transformação**: Normalização, padronização, codificação de categóricas
4. **Engenharia de features**: Criação de novas variáveis, seleção de features
5. **Modelagem**: Treinamento e otimização do modelo
6. **Avaliação**: Medição do desempenho e validação
7. **Implantação**: Colocação do modelo em produção

Neste módulo, focaremos principalmente nos componentes de pré-processamento (etapas 2-4), e abordaremos também aspectos da avaliação e organização de projetos.

## 4. Pipelines no scikit-learn

O scikit-learn fornece a classe `Pipeline` que facilita a construção e execução de pipelines de pré-processamento. Suas principais características:

- Encadeia múltiplos passos de processamento
- Aplica cada etapa sequencialmente
- Expõe os mesmos métodos (fit, transform, predict) que os estimadores individuais
- Gerencia automaticamente a passagem de dados entre etapas

## 5. Exemplo Simples de Pipeline

Vamos começar com um exemplo básico para entender a sintaxe e funcionamento de um pipeline no scikit-learn:

In [ ]:
# @title
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# MODIFICAÇÃO 1: Adicionando outliers artificiais ao dataset
# Isso fará com que a normalização seja muito afetada pelo vazamento
np.random.seed(42)

# Identificando um subconjunto de dados para ser "teste"
test_mask = np.zeros(len(X), dtype=bool)
test_mask[np.random.choice(len(X), size=int(len(X)*0.3), replace=False)] = True

# Adicionando outliers extremos apenas nas amostras que serão de teste
outlier_scale = 10  # Fator de escala para os outliers
for col in X.columns[:5]:  # Modificando apenas algumas colunas
    # Multiplicando os valores no conjunto de teste por um fator grande
    X.loc[test_mask, col] = X.loc[test_mask, col] * outlier_scale

# MODIFICAÇÃO 2: Adicionando valores ausentes de forma não aleatória
# Isso tornará a imputação mais sensível ao vazamento
missing_proportion = 0.2
for col in X.columns[5:10]:  # Escolhendo outras colunas para valores ausentes
    # Criando valores ausentes nos menores valores da coluna
    threshold = X[col].quantile(missing_proportion)
    X.loc[(X[col] <= threshold) & ~test_mask, col] = np.nan

print(f"Dataset modificado: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"Valores ausentes: {X.isna().sum().sum()}")

print(f"Dataset carregado: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"Valores ausentes: {X.isna().sum().sum()}")

In [ ]:

# Criando um pipeline simples com duas etapas
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # Etapa 1: Imputação de valores ausentes
    ('scaler', StandardScaler())                  # Etapa 2: Padronização dos dados
])

# Criando dados de exemplo
X = np.array([
    [1, 2, np.nan],
    [4, np.nan, 6],
    [np.nan, 8, 9],
    [10, 11, 12]
])

# Aplicando o pipeline
X_transformed = pipe.fit_transform(X)

print("Dados originais:")
print(X)
print("\nDados transformados:")
print(X_transformed)

### O que aconteceu no exemplo acima?

1. Criamos um pipeline com duas etapas: `SimpleImputer` e `StandardScaler`
2. O pipeline processou o array X sequencialmente:
   - Primeiro, o `SimpleImputer` substituiu os valores ausentes (NaN) pela média de cada coluna
   - Em seguida, o `StandardScaler` padronizou os dados para média 0 e desvio padrão 1
3. O método `fit_transform()` ajustou os transformadores aos dados e depois aplicou a transformação

É importante notar que o pipeline mantém os parâmetros aprendidos (como médias e desvios padrão) para uso consistente em novos dados.

## 6. Comparação: Manual vs Pipeline

Vamos comparar duas abordagens: transformações manuais e usando pipeline.

### Abordagem 1: Transformações manuais

In [ ]:
# Dados de exemplo
X = np.array([
    [1, 2, np.nan],
    [4, np.nan, 6],
    [np.nan, 8, 9],
    [10, 11, 12]
])

# Divisão em treino e teste
X_train, X_test = train_test_split(X, test_size=0.5, random_state=42)

# Abordagem manual - Etapas aplicadas separadamente
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

# Ajustando e transformando os dados de treino
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transformando os dados de teste com os mesmos parâmetros
X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

print("Dados de treino transformados manualmente:")
print(X_train_scaled)
print("\nDados de teste transformados manualmente:")
print(X_test_scaled)

### Abordagem 2: Usando Pipelines

In [ ]:
# Mesmo conjunto de dados e divisão
X = np.array([
    [1, 2, np.nan],
    [4, np.nan, 6],
    [np.nan, 8, 9],
    [10, 11, 12]
])

X_train, X_test = train_test_split(X, test_size=0.5, random_state=42)

# Abordagem com pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Ajustando no treino e transformando treino e teste
X_train_pipeline = pipeline.fit_transform(X_train)
X_test_pipeline = pipeline.transform(X_test)

print("Dados de treino transformados com pipeline:")
print(X_train_pipeline)
print("\nDados de teste transformados com pipeline:")
print(X_test_pipeline)

# Verificando se os resultados são os mesmos
print("\nOs resultados são idênticos?")
print("Treino:", np.allclose(X_train_scaled, X_train_pipeline))
print("Teste:", np.allclose(X_test_scaled, X_test_pipeline))

### Vantagens da abordagem com Pipelines:

1. **Código mais limpo e conciso:** Menos linhas de código, menos variáveis intermediárias
2. **Menor chance de erros:** Não há risco de aplicar as transformações na ordem errada
3. **Maior organização:** As etapas de processamento estão claramente definidas
4. **Facilidade de manutenção:** Para adicionar ou remover etapas, basta modificar a definição do pipeline
5. **Tratamento consistente:** Garantia de que as mesmas transformações são aplicadas aos dados de teste

Nas próximas partes, veremos como os pipelines nos ajudam a prevenir um problema comum em ciência de dados: o vazamento de dados (data leakage) e explorar como processar diferentes tipos de variáveis.